<a href="https://colab.research.google.com/github/tpopples/Data4GoodatCornell/blob/main/sessions/[Session 2] Big Red Lining!/D4G_Session_2_Redlining.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"/>
</a>

In [2]:
%config InlineBackend.figure_formats = ["retina"]

In [3]:
!pip install -q geopandas seaborn folium census


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


# Redlining

In [4]:
import folium # SLIDES
import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from branca.utilities import color_brewer
from census import Census

In [5]:
census = Census("", year=2020)

In [6]:
areas_gdf = gpd.read_file("https://dsl.richmond.edu/panorama/redlining/static/mappinginequality.gpkg") # SLIDES

In [7]:
areas_gdf

,area_id,city,state,city_survey,category,grade,label,residential,commercial,industrial,fill,geometry
0,244,Birmingham,AL,True,Best,A,A1,True,False,False,#76a865,"MULTIPOLYGON (((-86.75678 33.49754, -86.75653 ..."
1,193,Birmingham,AL,True,Best,A,A2,True,False,False,#76a865,"MULTIPOLYGON (((-86.75867 33.50933, -86.76134 ..."
2,206,Birmingham,AL,True,Best,A,A3,True,False,False,#76a865,"MULTIPOLYGON (((-86.75678 33.49754, -86.75692 ..."
3,203,Birmingham,AL,True,Still Desirable,B,B1,True,False,False,#7cb5bd,"MULTIPOLYGON (((-86.80111 33.48071, -86.80505 ..."
4,189,Birmingham,AL,True,Still Desirable,B,B10,True,False,False,#7cb5bd,"MULTIPOLYGON (((-86.74923 33.53333, -86.74971 ..."
...,...,...,...,...,...,...,...,...,...,...,...,...
10149,6518,Wheeling,WV,True,Hazardous,D,D4,True,False,False,#d9838d,"MULTIPOLYGON (((-80.71985 40.06376, -80.71943 ..."
10150,6520,Wheeling,WV,True,Hazardous,D,D5,True,False,False,#d9838d,"MULTIPOLYGON (((-80.72854 40.06729, -80.7287 4..."
10151,6516,Wheeling,WV,True,Hazardous,D,D6,True,False,False,#d9838d,"MULTIPOLYGON (((-80.72216 40.06134, -80.72247 ..."
10152,6512,Wheeling,WV,True,Hazardous,D,D7,True,False,False,#d9838d,"MULTIPOLYGON (((-80.6474 40.04886, -80.64842 4..."


In [8]:
queens_bronx_areas_gdf = areas_gdf[(areas_gdf["city"] == "Queens") | (areas_gdf["city"] == "Bronx")].copy()
queens_bronx_areas_gdf["score"] = queens_bronx_areas_gdf["grade"].map({"A": 3, "B": 2, "C": 1, "D": 0})
queens_bronx_areas_gdf

,area_id,city,state,city_survey,category,grade,label,residential,commercial,industrial,fill,geometry,score
6271,2736,Bronx,NY,True,Best,A,A1,True,False,False,#76a865,"MULTIPOLYGON (((-73.90345 40.91062, -73.90615 ...",3.0
6272,2738,Bronx,NY,True,Best,A,A2,True,False,False,#76a865,"MULTIPOLYGON (((-73.89916 40.89286, -73.89949 ...",3.0
6273,2737,Bronx,NY,True,Still Desirable,B,B1,True,False,False,#7cb5bd,"MULTIPOLYGON (((-73.90652 40.9006, -73.90522 4...",2.0
6274,2764,Bronx,NY,True,Still Desirable,B,B10,True,False,False,#7cb5bd,"MULTIPOLYGON (((-73.85101 40.85032, -73.84922 ...",2.0
6275,2775,Bronx,NY,True,Still Desirable,B,B11,True,False,False,#7cb5bd,"MULTIPOLYGON (((-73.83816 40.85735, -73.83685 ...",2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
6812,2472,Queens,NY,True,Hazardous,D,D6,True,False,False,#d9838d,"MULTIPOLYGON (((-73.85912 40.75063, -73.86101 ...",0.0
6813,2473,Queens,NY,True,Hazardous,D,D7,True,False,False,#d9838d,"MULTIPOLYGON (((-73.85262 40.75252, -73.85424 ...",0.0
6814,2475,Queens,NY,True,Hazardous,D,D8,True,False,False,#d9838d,"MULTIPOLYGON (((-73.85494 40.74337, -73.8572 4...",0.0
6815,2474,Queens,NY,True,Hazardous,D,D9,True,False,False,#d9838d,"MULTIPOLYGON (((-73.85494 40.74337, -73.85476 ...",0.0


In [9]:
queens_bronx_areas_gdf["grade"].value_counts()

grade
C    134
D     46
B     28
A      3
E      1
Name: count, dtype: int64

In [10]:
queens_bronx_map = folium.Map([40.65, -73.95], zoom_start=12)
queens_bronx_choro = folium.Choropleth(
    geo_data=queens_bronx_areas_gdf,
    name="choropleth",
    data=queens_bronx_areas_gdf,
    columns=["area_id", "score"],
    key_on="feature.properties.area_id",
    fill_color="RdYlGn",
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name="Historic Redlining Indicator",
)
queens_bronx_choro.color_scale.colors = list(reversed(queens_bronx_choro.color_scale.colors))
queens_bronx_choro.add_to(queens_bronx_map)
queens_bronx_map

In [11]:
!wget -O HRI2020.zip "https://raw.githubusercontent.com/tpopples/Data4GoodatCornell/main/data/HRI2020.zip"

zsh:1: command not found: wget


In [12]:
hri_gdf = gpd.read_file("HRI2020.zip")

DataSourceError: '/vsizip/HRI2020.zip' not recognized as being in a supported file format. It might help to specify the correct driver explicitly by prefixing the file path with '<DRIVER>:', e.g. 'CSV:path'.

In [ ]:
hri_gdf

In [ ]:
metro_areas = {
    "NYC": hri_gdf[hri_gdf["METRO_NAME"] == "New York-Newark-Jersey City, N"],
    "LA": hri_gdf[hri_gdf["METRO_NAME"] == "Los Angeles-Long Beach, CA"],
    "Houston": hri_gdf[hri_gdf["GEOID20"].str.startswith("48201")]
}

In [ ]:
def plot_hri(gdf, center, zoom):
    hri_map = folium.Map(center, zoom_start=zoom)

    # ensure colors
    def reverse_color_brewer(*args, **kwargs):
        return list(reversed(color_brewer(*args, **kwargs)))

    folium.features.color_brewer = reverse_color_brewer
    choro = folium.Choropleth(
        geo_data=gdf,
        name="choropleth",
        data=gdf,
        columns=["GEOID20", "HRI2020"],
        key_on="feature.properties.GEOID20",
        fill_color="RdYlGn",
        fill_opacity=0.7,
        line_opacity=0.2,
        legend_name="Historic Redlining Indicator",
    )
    folium.features.color_brewer = color_brewer
    choro.add_to(hri_map)
    return hri_map

In [ ]:
plot_hri(metro_areas["NYC"], center=[40.65, -73.95], zoom=12)

In [ ]:
plot_hri(metro_areas["LA"], center=[34.06, -118.25], zoom=11)

In [ ]:
plot_hri(metro_areas["Houston"], center=[34.06, -118.25], zoom=11)